# Bagging regressor
Single regressors against BaggingRegressor on the Boston housing data, then a grid search over the bagging
settings. The curve demo is `app.py` (`python app.py`, then open http://127.0.0.1:8050).

In [1]:
import time

import pandas as pd
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

## 1. The data
`load_boston` was removed from scikit-learn in 1.2; the same table is in data/boston.csv (OpenML dataset 531).

In [2]:
boston = pd.read_csv("data/boston.csv")
X = boston.drop(columns="MEDV")
y = boston["MEDV"]
print("inputs:", list(X.columns))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)
print(X_train.shape, X_test.shape)

inputs: ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']
(404, 13) (102, 13)


## 2. Three single regressors

In [3]:
models = {"linear regression": LinearRegression(), "decision tree": DecisionTreeRegressor(random_state=1),
          "KNN": KNeighborsRegressor()}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}: test R² {r2_score(y_test, model.predict(X_test)):.3f}")

linear regression: test R² 0.659
decision tree: test R² 0.380


KNN: test R² 0.548


## 3. A bagging regressor with every setting at its default
Default: 10 decision trees, each on a bootstrap sample of all the training rows, all columns.

In [4]:
bag = BaggingRegressor(random_state=1)
bag.fit(X_train, y_train)
print(f"training R² {bag.score(X_train, y_train):.3f}")
print(f"test R²     {bag.score(X_test, y_test):.3f}")

training R² 0.980
test R²     0.828


## 4. Grid search: base model, number of models, row and column sampling
`estimator=None` means a decision tree. 3 x 3 x 2 x 2 x 2 x 2 = 144 combinations, 3-fold cross-validation.

In [5]:
params = {
    "estimator": [None, LinearRegression(), KNeighborsRegressor()],
    "n_estimators": [20, 50, 100],
    "max_samples": [0.5, 1.0],
    "max_features": [0.5, 1.0],
    "bootstrap": [True, False],
    "bootstrap_features": [True, False],
}
start = time.time()
grid = GridSearchCV(BaggingRegressor(random_state=1), param_grid=params, cv=3, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
print(f"took {time.time() - start:.0f} s")
print(f"training R² {grid.best_estimator_.score(X_train, y_train):.3f}")
print(f"test R²     {grid.best_estimator_.score(X_test, y_test):.3f}")
print(f"best cross-validation R² {grid.best_score_:.3f}")
print("best parameters:", grid.best_params_)

Fitting 3 folds for each of 144 candidates, totalling 432 fits


took 43 s
training R² 0.983
test R²     0.806
best cross-validation R² 0.871
best parameters: {'bootstrap': True, 'bootstrap_features': False, 'estimator': None, 'max_features': 1.0, 'max_samples': 1.0, 'n_estimators': 50}


## 5. Out-of-bag R²: the regressor also has oob_score

In [6]:
bag_oob = BaggingRegressor(n_estimators=50, oob_score=True, random_state=1).fit(X_train, y_train)
print(f"out-of-bag R² {bag_oob.oob_score_:.3f}, test R² {bag_oob.score(X_test, y_test):.3f}")

out-of-bag R² 0.870, test R² 0.806


## 6. The curve demo (same code as app.py)

In [7]:
from app import figure, fit

(single, r1), (bagged, r2) = fit("decision tree", n_estimators=50, max_samples=25)
figure([(f"single tree: {r1:.2f}", single, "#E45756"), (f"bagging: {r2:.2f}", bagged, "#4C78A8")])

In [8]:
# One 102-observation test score is noisy. Repeat the split 100 times and average: every model on the same splits.
import numpy as np
splits = [train_test_split(X, y, test_size=0.2, random_state=s) for s in range(100)]
makers = {"linear regression": LinearRegression, "decision tree": lambda: DecisionTreeRegressor(random_state=1),
          "KNN": KNeighborsRegressor, "bagging, default": lambda: BaggingRegressor(random_state=1),
          "bagging, tuned (50 trees)": lambda: BaggingRegressor(n_estimators=50, random_state=1)}
scores = {n: np.array([m().fit(a, c).score(b, d) for a, b, c, d in splits]) for n, m in makers.items()}
for n, s in scores.items():
    print(f"{n}: mean test R² {s.mean():.3f}  (min {s.min():.3f}, max {s.max():.3f})")
tuned, default = scores["bagging, tuned (50 trees)"], scores["bagging, default"]
print(f"tuned better than default on {(tuned > default).sum()} of 100 splits")

linear regression: mean test R² 0.708  (min 0.506, max 0.830)
decision tree: mean test R² 0.716  (min -0.018, max 0.857)
KNN: mean test R² 0.504  (min 0.251, max 0.718)
bagging, default: mean test R² 0.841  (min 0.648, max 0.922)
bagging, tuned (50 trees): mean test R² 0.857  (min 0.629, max 0.938)
tuned better than default on 79 of 100 splits
